# Whole genome sequencing data extraction, preprocessing and quality control

## Notes before running the scripts
- Remember to replace all data and working directory paths with your actual paths.
- Depedencies: bcftools, plink2, remember to set the path of tools with your actual paths
- download snp vcf file at https://ftp.ncbi.nlm.nih.gov/snp/organisms/human_9606/VCF/All_20180418.vcf.gz


In [ ]:
genomic_data_path = "../../Data_from_All_of_Us_Controlled_tier/vwb-aou-datasets-controlled" # set your genotype data path
!ls "{genomic_data_path}"

In [ ]:
import subprocess
import os

In [ ]:
def filter_out_sample(chr_num, input_dir, cohort_pid_f, output_dir, thread_n):

    chr_prefix = f"acaf_threshold.chr{chr_num}"
    input_prefix = f"{input_dir}/{chr_prefix}"
    output_prefix = f"{output_dir}/{chr_prefix}"
    
    bgen_prefix = f"{output_dir}/bgen_chr{chr_num}"
  
    command = [
        "/home/jupyter/workspace/tools/plink2",  
        "--pfile", input_prefix,
        "--threads", str(thread_n),
        "--keep", cohort_pid_f,
        "--make-pgen",
        "--out", output_prefix
    ]
    subprocess.run(command)

In [ ]:
def run_plink_qc(chr_num, input_dir, output_dir, thread_n, maf=0.005, geno=0.1, hwe=1e-5, k=0.001, mind=0.1): #
    os.makedirs(output_dir, exist_ok=True)
    
    chr_prefix = f"acaf_threshold.chr{chr_num}"
    input_prefix = f"{input_dir}/{chr_prefix}"
    output_prefix = f"{output_dir}/{chr_prefix}_qc"
    
    bgen_prefix = f"{output_dir}/bgen_chr{chr_num}"
  
    command = [
        "/home/jupyter/workspace/tools/plink2",
        "--write-samples",
        "--max-alleles", "2",  
        "--autosome",          
        "--pfile", input_prefix,
        "--threads", str(thread_n),
        "--maf", str(maf),
        "--geno", str(geno),
        "--hwe", str(hwe), str(k),
        "keep-fewhet",
        "--mind", str(mind),
        "--make-bed",
        "--out", output_prefix
    ]
    subprocess.run(command)

In [ ]:
import os
import pandas as pd


def pheno_preprocessing(prefix, pheno_f="./data/phenotype/phenotype_all.csv"):

    backup_psam = f"{prefix}_backup.psam"
    output_psam = f"{prefix}.psam"

    # Backup original psam
    os.system(f"cp {prefix}.psam {backup_psam}")

    # -------------------------
    # Load phenotype file
    # -------------------------
    pheno = pd.read_csv(pheno_f)

    pheno = pheno[['FID', 'IID', 'Sex']].rename(
        columns={"Sex": "SEX"}
    )

    # Ensure IDs are strings
    pheno['FID'] = pheno['FID'].astype(str)
    pheno['IID'] = pheno['IID'].astype(str)

    # PLINK2 SEX encoding must be 0/1/2
    pheno['SEX'] = (
        pheno['SEX']
        .map({
            'Male': '1',
            'Female': '2',
            'M': '1',
            'F': '2'
        })
        .fillna('0')
    )

    psam = pd.read_csv(backup_psam, sep="\t")

    if '#IID' in psam.columns:
        psam = psam.rename(columns={'#IID': 'IID'})

    # Create FID if absent
    if 'FID' not in psam.columns and '#FID' not in psam.columns:
        psam['FID'] = psam['IID']

    # Normalize column names
    if '#FID' in psam.columns:
        psam = psam.rename(columns={'#FID': 'FID'})

    psam['FID'] = psam['FID'].astype(str)
    psam['IID'] = psam['IID'].astype(str)

    # Ensure SEX column exists
    if 'SEX' not in psam.columns:
        psam['SEX'] = '0'

    psam['SEX'] = psam['SEX'].fillna('0').astype(str)

    # -------------------------
    # Merge phenotype SEX info
    # -------------------------
    psam = psam.merge(
        pheno[['FID', 'IID', 'SEX']],
        on=['FID', 'IID'],
        how='left',
        suffixes=('', '_new')
    )

    # Replace SEX with updated values
    psam['SEX'] = psam['SEX_new'].fillna(psam['SEX'])

    psam = psam.drop(columns=['SEX_new'])

    # Final cleanup
    psam['SEX'] = psam['SEX'].fillna('0').astype(str)

    # Restore PLINK column names
    psam = psam.rename(columns={'FID': '#FID'})

    # -------------------------
    # Debug
    # -------------------------
    print(psam['SEX'].value_counts(dropna=False))

    # -------------------------
    # Save
    # -------------------------
    psam = psam[['#FID', 'IID', 'SEX']]

    psam.to_csv(
        output_psam,
        sep='\t',
        index=False
    )

In [ ]:
def run_tasks(chr_no, raw_genotype_dir, cohort_pid_f, pheno_f,cohort_pgen_dir, qc_genotype_dir,thread_n):

    filter_out_sample(chr_no, raw_genotype_dir, cohort_pid_f, cohort_pgen_dir, thread_n)
    pheno_preprocessing(f"{cohort_pgen_dir}/acaf_threshold.chr{chr_no}", pheno_f)
    run_plink_qc(chr_no, cohort_pgen_dir, qc_genotype_dir, thread_n)

In [ ]:
thread_n=16

raw_genotype_dir = f"{genomic_data_path}/v8/wgs/short_read/snpindel/acaf_threshold/pgen"

pheno_f="/working_directory/phenotype_all.csv"
cohort_pid_f = "/working_directory/keep_samples.txt"

cohort_pgen_dir = "/working_directory/pgen_cohort"

qc_genotype_dir = "/working_directory/pgen_qc_maf" 

chromosomes = [str(c) for c in range(21, 23)]

for chr_label in chromosomes:
    run_tasks(chr_label, raw_genotype_dir, cohort_pid_f, pheno_f,cohort_pgen_dir,qc_genotype_dir, thread_n)

In [ ]:
import pandas as pd
import subprocess
import os
import re

# replace /opt/conda/envs/pgs_env/bin/bcftools with your actual bcftools path

def map_bim_snps_with_bcftools(original_prefix, vcf_path, output_dir, allow_flip=True):

    prefix = original_prefix.split('/')[-1]
    bim_path = original_prefix + ".bim"

    match = re.search(r"chr(\d+)", bim_path)
    if not match:
        raise ValueError(f"Cannot extract chromosome from {bim_path}")
    chr_label = match.group(1)
    print(f"Processing chromosome: {chr_label}")

    snp_map_path = bim_path + ".snp_mapping"

    extract_cmd = (
        f"/opt/conda/envs/pgs_env/bin/bcftools view -r {chr_label} -m2 -M2 {vcf_path} -Ou | "
        f"/opt/conda/envs/pgs_env/bin/bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%ID\n' > {snp_map_path}"
    )
    subprocess.run(extract_cmd, shell=True, check=True)

    bim = pd.read_csv(bim_path, sep=r'\s+', header=None,
                      names=['chr', 'snp', 'cm', 'pos', 'a1', 'a2'])
    snp_map = pd.read_csv(snp_map_path, sep='\t', header=None,
                          names=['chr', 'pos', 'ref', 'alt', 'rsid'])

    bim['chr'] = bim['chr'].astype(str)
    snp_map['chr'] = snp_map['chr'].astype(str)
    bim['pos'] = bim['pos'].astype(int)
    snp_map['pos'] = snp_map['pos'].astype(int)

    bim = bim.reset_index()
    merged = pd.merge(bim, snp_map, on=['chr', 'pos'], how='left')

    def alleles_match(row):
        a1, a2 = row['a1'].upper(), row['a2'].upper()
        ref, alt = str(row['ref']).upper(), str(row['alt']).upper()

        if pd.isna(ref) or pd.isna(alt):
            return False

        match_direct = (a1, a2) == (ref, alt) or (a1, a2) == (alt, ref)
        match_flip = False
        if allow_flip:
            flip = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
            try:
                a1f = ''.join(flip.get(b, '') for b in a1)
                a2f = ''.join(flip.get(b, '') for b in a2)
                match_flip = (a1f, a2f) == (ref, alt) or (a1f, a2f) == (alt, ref)
            except Exception:
                match_flip = False

        return match_direct or match_flip

    merged['match'] = merged.apply(alleles_match, axis=1)

    # Assign rsID where available, fallback to chr_pos_a1_a2 where not
    def assign_snp_id(row):
        if not row['match']:
            return f"{row['chr']}_{row['pos']}_{row['a1'].upper()}_{row['a2'].upper()}"
        if pd.notnull(row['rsid']) and row['rsid'] != ".":
            return row['rsid']
        
    merged['new_snp'] = merged.apply(assign_snp_id, axis=1)
    bim['snp'] = merged['new_snp']
    
    new_bim = os.path.join(output_dir, prefix + ".bim")
    
    #print(bim)
    bim = bim[['chr', 'snp', 'cm', 'pos', 'a1', 'a2']]
    bim.to_csv(new_bim, sep='\t', header=False, index=False)

    matched = merged[merged['match']]
    
    pct = matched.shape[0] / bim.shape[0] * 100
    print(pct)
    print(f"{prefix} matched SNPs: {matched.shape[0]} / {bim.shape[0]} = {pct:.2f}%")


In [ ]:
# replace the following snp_vcf_path or working_directory path with your actual path

import multiprocessing as mp

def run_mapping(chr_n):
    
    plink_prefix = f"/working_directory/pgen_qc_maf/acaf_threshold.chr{chr_n}_qc"
    
    final_output_dir = "/working_directory/bed_qc_rsid"
    os.makedirs(final_output_dir, exist_ok=True)

    dbsnp_vcf = "/snp_vcf_path/All_20180418.vcf.gz"  # GRCh38 build

    map_bim_snps_with_bcftools(plink_prefix, dbsnp_vcf, final_output_dir, allow_flip=True)

In [ ]:
chroms = list(range(21, 23))  

with mp.Pool(processes=1) as pool:  # Adjust number of processes as needed
    pool.map(run_mapping, chroms)

In [ ]:
import os
import subprocess

def merge_bfiles(input_dir, output_dir, chr_list=range(1,23),
                       merged_prefix="merged"):
    """
    Merge per-chromosome PLINK1 .bfile files (already containing RSIDs) into one genome-wide .bfile
    and convert to PLINK2 .pgen format. Automatically handles merge failures via .missnp.

    Parameters:
    - input_dir: directory containing per-chromosome .bed/.bim/.fam files
    - output_dir: directory to save merged and pgen files
    - chr_list: list of chromosomes to merge (default 1-23)
    - merged_prefix: output prefix
    """
    os.makedirs(output_dir, exist_ok=True)
    processed_prefixes = []

    # Step 0: List per-chromosome files
    for chr_num in chr_list:
        chr_prefix = os.path.join(input_dir, f"acaf_threshold.chr{chr_num}_qc")
        processed_prefixes.append(chr_prefix)

    # Step 1: Merge chromosomes
    merge_list_file = os.path.join(output_dir, "merge_list.txt")
    with open(merge_list_file, "w") as f:
        for chr_prefix in processed_prefixes[1:]:
            f.write(chr_prefix + "\n")

    base_chr_prefix = processed_prefixes[0]
    merged_bfile_prefix = os.path.join(output_dir, merged_prefix)
    merge_cmd = [
        "/home/jupyter/workspace/tools/plink",
        "--bfile", base_chr_prefix,
        "--merge-list", merge_list_file, 
        "--make-bed",
        "--out", merged_bfile_prefix
    ]

    print("Attempting to merge per-chromosome .bfiles...")
    try:
        subprocess.run(merge_cmd, check=True)
        print(f"Merged .bfile created: {merged_bfile_prefix}.bed/.bim/.fam")
    except subprocess.CalledProcessError:
        print("Merge failed! Attempting to fix using .missnp...")
        missnp_file = merged_bfile_prefix + "-merge.missnp"
        if os.path.exists(missnp_file):
            print(f"Excluding {missnp_file} and retrying merge...")
            for i, prefix in enumerate(processed_prefixes):
                cleaned_prefix = prefix + "_nomissnp"
                cmd_exclude = [
                    "/home/jupyter/workspace/tools/plink",
                    "--bfile", prefix,
                    "--exclude", missnp_file,
                    "--make-bed",
                    "--out", cleaned_prefix
                ]
                subprocess.run(cmd_exclude, check=True)
                processed_prefixes[i] = cleaned_prefix

            # Rewrite merge list
            with open(merge_list_file, "w") as f:
                for chr_prefix in processed_prefixes[1:]:
                    f.write(chr_prefix + "\n")

            # Retry merge
            merge_cmd = [
                "/home/jupyter/workspace/tools/plink",
                "--bfile", processed_prefixes[0],
                "--merge-list", merge_list_file,
                "--make-bed",
                "--out", merged_bfile_prefix
            ]
            subprocess.run(merge_cmd, check=True)
            print(f"Merged .bfile created after removing problematic SNPs: {merged_bfile_prefix}.bed/.bim/.fam")
        else:
            raise FileNotFoundError(f"Merge failed but .missnp file not found: {missnp_file}")


In [ ]:
input_dir = "/working_directory/bed_qc_rsid"
output_dir = "/working_directory/qc_merged"
merge_bfiles(input_dir, output_dir, chr_list=range(1,23), merged_prefix="merged_genome")

Performing 3-pass merge (18300 people, 4255768/8752072 variants per pass).
Pass 1 complete.                              
Pass 2 complete.                              
Merged fileset written to ./data/pgen_qc_merged/merged_genome-merge.bed +
./data/pgen_qc_merged/merged_genome-merge.bim +
./data/pgen_qc_merged/merged_genome-merge.fam .
8752072 variants loaded from .bim file.
18300 people (5209 males, 12891 females, 200 ambiguous) loaded from .fam.
Ambiguous sex IDs written to ./data/pgen_qc_merged/merged_genome.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 18300 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is 0.998567.
8752072 variants and 18300 people pass filters and QC.
Note: No phenotypes present.
--make-b